<a href="https://colab.research.google.com/github/brasilemnumeros/dados/blob/main/indicadores/pib/Investimento.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Taxa de investimento — formação bruta de capital fixo (% do PIB)

Este notebook é o motor de coleta e publicação da série `taxa-investimento-pib`.

- **Fonte:** IBGE — Sistema de Contas Nacionais, referência 2010, lido na API OData do
  IPEADATA, que transcreve as séries do IBGE. Numerador: **FBCF a preços correntes**
  (`SCN10_FBKFN10`, R$ milhões). Denominador: **PIB a preços de mercado** (`SCN10_PIBN10`,
  R$ milhões).
- **Endpoints:**
  `https://www.ipeadata.gov.br/api/odata4/ValoresSerie(SERCODIGO='SCN10_FBKFN10')` e
  `https://www.ipeadata.gov.br/api/odata4/ValoresSerie(SERCODIGO='SCN10_PIBN10')`.
- **Definição:** formação bruta de capital fixo — máquinas, equipamentos, construção e outros
  bens de capital — como proporção do PIB a preços correntes do mesmo ano. É a “taxa de
  investimento”. `percentual` (`%`, 1 casa), `eixo: percentual`, `medida: participacao`,
  granularidade `anual`, data em `AAAA-01-01`.
- **Fórmula:** `taxa = 100 × FBCF / PIB`, arredondada a 1 casa com meio para cima. Os dois
  valores são do mesmo ano e a preços correntes; não há deflação.
- **Período publicado:** de `1995-01-01` até o último ano fechado disponível.
- **Arquivo gerado:** `indicadores/pib/taxa-investimento-pib.json`, com `versao: 1` e
  `id: taxa-investimento-pib`.

## O que o número é — e o que não é

- **É um fluxo do ano, não um estoque.** Mede quanto se investiu no ano, não o capital
  acumulado nem a dívida.
- **É FBCF, não “investimento” financeiro.** Não inclui aplicação em títulos, ações ou
  poupança; é a compra de bens de capital que ampliam a capacidade de produzir.
- **É a preços correntes.** Numerador e denominador do mesmo ano; a comparação entre anos
  embute mudança de preços relativos.
- **Não é a taxa de poupança.** Poupança e investimento são conceitos próximos, mas a taxa de
  investimento é a FBCF sobre o PIB; a poupança bruta é outra série.
- **O denominador muda com a revisão do PIB.** Quando o IBGE revisa as Contas Nacionais, a
  razão inteira pode mudar; é revisão da fonte, não erro de fórmula.
- **A série concatena compilações do IBGE.** O metadado do IPEADATA declara: 1995–1999 são
  estimativas a partir das Contas Nacionais Trimestrais (referência 2010); 2000–2021 vêm do
  Sistema de Contas Nacionais (referência 2010); 2022–2023 são resultados preliminares
  estimados a partir das Trimestrais; 2024 está na tabela anual; 2025 é a primeira estimativa
  anual do fechamento do 4º trimestre. A definição é idêntica em todos os trechos; muda a
  compilação.

## Coleta das séries oficiais

A leitura é feita desde o início da série, sem atualização incremental e sem ler o JSON já
publicado para decidir o período. O IPEADATA devolve `VALDATA` em ISO com fuso e `VALVALOR` em
R$ milhões; o notebook usa o ano e não arredonda a leitura.

Além das duas séries do numerador e do denominador, o notebook busca o PIB a preços correntes
na tabela **6784** do SIDRA (variável **9808**) para conferir o denominador contra o canal
oficial do IBGE.

In [1]:
from datetime import date
from decimal import Decimal, ROUND_HALF_UP
from pathlib import Path
import json
import math
import os
import time

import requests

RAIZ = Path(os.environ.get("DADOS_RAIZ", Path.cwd())).resolve()
PASTA = RAIZ / "indicadores" / "pib"
if not PASTA.is_dir():
    PASTA = RAIZ

DESTINO = Path(os.environ.get("PIB_SAIDA", PASTA)).resolve()
ARQUIVO = DESTINO / "taxa-investimento-pib.json"


def relativo(caminho):
    """Caminho exibido no notebook: relativo à raiz, nunca o absoluto da máquina."""
    try:
        return caminho.relative_to(RAIZ)
    except ValueError:
        return caminho.name

In [2]:
BASE_IPEADATA = "https://www.ipeadata.gov.br/api/odata4/ValoresSerie(SERCODIGO='{codigo}')"
CODIGO_FBCF = "SCN10_FBKFN10"
CODIGO_PIB = "SCN10_PIBN10"
URL_SIDRA = "https://apisidra.ibge.gov.br/values/t/6784/n1/all/v/9808/p/all?formato=json"
HOJE = date.today()
PRIMEIRO_ANO = 1995


def baixar_json(url, tentativas=4, timeout=120):
    """GET com espera crescente; IPEADATA e SIDRA falham de forma intermitente."""
    for tentativa in range(tentativas):
        try:
            resposta = requests.get(url, timeout=timeout)
            resposta.raise_for_status()
            return resposta.json()
        except (requests.RequestException, ValueError):
            if tentativa == tentativas - 1:
                raise
            time.sleep(2 ** tentativa)


def ler_ipeadata(codigo):
    """Série anual do IPEADATA: {ano: valor em R$ milhões}."""
    bruto = baixar_json(BASE_IPEADATA.format(codigo=codigo))
    if not isinstance(bruto, dict) or "value" not in bruto:
        raise ValueError(f"{codigo}: resposta fora do formato OData")
    serie = {}
    for item in bruto["value"]:
        if item.get("VALVALOR") is None:
            continue
        data = item["VALDATA"]
        if data[5:10] != "01-01":
            raise ValueError(f"{codigo}: data fora de 1º de janeiro ({data})")
        ano = int(data[:4])
        if ano in serie:
            raise ValueError(f"{codigo}: ano repetido ({ano})")
        serie[ano] = float(item["VALVALOR"])
    if not serie:
        raise ValueError(f"{codigo}: série vazia")
    return serie


fbcf = ler_ipeadata(CODIGO_FBCF)
pib = ler_ipeadata(CODIGO_PIB)

bruto_sidra = baixar_json(URL_SIDRA)
if not isinstance(bruto_sidra, list) or len(bruto_sidra) < 2:
    raise ValueError("SIDRA 6784: resposta fora do formato de lista")
sidra = {}
for linha in bruto_sidra[1:]:
    if linha["V"] in ("..", "...", "-", "X"):
        continue
    sidra[int(linha["D3C"])] = float(linha["V"])

print(f"FBCF ({CODIGO_FBCF}): {len(fbcf)} anos, {min(fbcf)} a {max(fbcf)}")
print(f"PIB  ({CODIGO_PIB}): {len(pib)} anos, {min(pib)} a {max(pib)}")
print(f"PIB no SIDRA 6784/9808: {len(sidra)} anos, {min(sidra)} a {max(sidra)}")
print(f"Arquivo de saída: {relativo(ARQUIVO)}")
print("\nÚltimos 3 anos do PIB (R$ milhões):")
for ano in sorted(pib)[-3:]:
    print(f"  {ano}  {pib[ano]:,.3f}")

FBCF (SCN10_FBKFN10): 79 anos, 1947 a 2025
PIB  (SCN10_PIBN10): 126 anos, 1900 a 2025
PIB no SIDRA 6784/9808: 29 anos, 1996 a 2024
Arquivo de saída: indicadores/pib/taxa-investimento-pib.json

Últimos 3 anos do PIB (R$ milhões):
  2023  10,943,344.668
  2024  11,779,250.582
  2025  12,738,565.614


## Cálculo da taxa

Os dois valores são do mesmo ano e a preços correntes, então a taxa é a razão direta. O corte
recente é o último **ano fechado** (`HOJE.year - 1`): o ano civil termina em dezembro e o IBGE
fecha a conta na divulgação do 4º trimestre, em março do ano seguinte. O corte inicial é 1995,
o primeiro ano da compilação trimestral.

A conferência do denominador compara o PIB do IPEADATA com o do SIDRA 6784 em todos os anos
que o SIDRA cobre. A tolerância de R$ 1 milhão absorve o arredondamento das duas publicações.

In [3]:
ANOS = sorted(set(fbcf) & set(pib))
LIMITE = HOJE.year - 1
ANOS = [ano for ano in ANOS if PRIMEIRO_ANO <= ano <= LIMITE]

if not ANOS:
    raise ValueError("nenhum ano fechado a partir de 1995")
if ANOS[0] != PRIMEIRO_ANO:
    raise ValueError(f"a fonte começa em {ANOS[0]}, não em {PRIMEIRO_ANO}")
if ANOS != list(range(ANOS[0], ANOS[-1] + 1)):
    faltando = sorted(set(range(ANOS[0], ANOS[-1] + 1)) - set(ANOS))
    raise ValueError(f"lacuna na série: {faltando}")
if ANOS[-1] < HOJE.year - 1:
    print(f"AVISO: a fonte não tem {HOJE.year - 1}; último ano fechado é {ANOS[-1]}")

for ano in ANOS:
    if fbcf[ano] <= 0 or pib[ano] <= 0:
        raise ValueError(f"{ano}: FBCF ou PIB não positivo ({fbcf[ano]}, {pib[ano]})")

# O denominador do IPEADATA tem de ser o mesmo PIB do canal oficial do IBGE.
conferidos = [ano for ano in ANOS if ano in sidra]
for ano in conferidos:
    if abs(pib[ano] - sidra[ano]) > 1.0:
        raise ValueError(f"{ano}: PIB IPEADATA {pib[ano]} x SIDRA {sidra[ano]}")
if len(conferidos) < 25:
    raise ValueError(f"poucos anos conferidos contra o SIDRA: {len(conferidos)}")

TAXA = {ano: 100 * fbcf[ano] / pib[ano] for ano in ANOS}


def arredondar(valor):
    """Arredonda para 1 casa com meio para cima (convenção comercial)."""
    return float(Decimal(repr(valor)).quantize(Decimal("0.1"), rounding=ROUND_HALF_UP))


print(f"Anos fechados: {len(ANOS)} ({ANOS[0]} a {ANOS[-1]})")
print(f"Denominador conferido contra o SIDRA 6784 em {len(conferidos)} anos "
      f"({conferidos[0]}–{conferidos[-1]}); 1995 e {ANOS[-1]} ficam fora.")
print("\nAno   FBCF (R$ mi)     PIB (R$ mi)   taxa   publicado")
for ano in ANOS:
    if ano not in (1995, 2003, 2013, 2017, 2020, 2024, 2025):
        continue
    print(f"{ano}  {fbcf[ano]:14,.3f}  {pib[ano]:15,.3f}  {TAXA[ano]:5.2f}%  {arredondar(TAXA[ano]):4.1f}%")

assert arredondar(TAXA[1995]) == 20.3, arredondar(TAXA[1995])
assert arredondar(TAXA[2003]) == 16.6, arredondar(TAXA[2003])
assert arredondar(TAXA[2013]) == 20.9, arredondar(TAXA[2013])
assert arredondar(TAXA[2017]) == 14.6, arredondar(TAXA[2017])
assert arredondar(TAXA[2020]) == 16.6, arredondar(TAXA[2020])

Anos fechados: 31 (1995 a 2025)
Denominador conferido contra o SIDRA 6784 em 29 anos (1996–2024); 1995 e 2025 ficam fora.

Ano   FBCF (R$ mi)     PIB (R$ mi)   taxa   publicado
1995     143,219.548      705,991.553  20.29%  20.3%
2003     285,261.518    1,717,950.396  16.60%  16.6%
2013   1,114,944.000    5,331,619.000  20.91%  20.9%
2017     958,779.000    6,585,479.000  14.56%  14.6%
2020   1,260,228.000    7,609,597.000  16.56%  16.6%
2024   1,991,796.042   11,779,250.582  16.91%  16.9%
2025   2,145,050.278   12,738,565.614  16.84%  16.8%


## Escrita do JSON canônico

Os pontos são gravados como `{"data": "AAAA-01-01", "valor": ...}`, em ordem ascendente, sem
campos extras. O valor é a taxa arredondada a 1 casa, a mesma precisão com que o IBGE publica a
taxa de investimento; `unidade.casas` é só exibição.

In [4]:
pontos = [
    {"data": f"{ano:04d}-01-01", "valor": arredondar(TAXA[ano])} for ano in ANOS
]

FONTE = (
    "IBGE (Contas Nacionais, via IPEADATA); "
    "taxa calculada pelo site (FBCF / PIB a preços correntes)"
)

serie = {
    "versao": 1,
    "id": "taxa-investimento-pib",
    "unidade": {"tipo": "percentual", "simbolo": "%", "casas": 1},
    "eixo": "percentual",
    "medida": "participacao",
    "granularidade": "anual",
    "fonte": FONTE,
    "dados": pontos,
}

DESTINO.mkdir(parents=True, exist_ok=True)
with ARQUIVO.open("w", encoding="utf-8") as arquivo:
    json.dump(serie, arquivo, ensure_ascii=False, indent=2)
    arquivo.write("\n")

print(f"{len(pontos)} pontos escritos em {relativo(ARQUIVO)}")
print(f"Primeiro ponto: {pontos[0]}")
print(f"Último ponto:   {pontos[-1]}")

31 pontos escritos em indicadores/pib/taxa-investimento-pib.json
Primeiro ponto: {'data': '1995-01-01', 'valor': 20.3}
Último ponto:   {'data': '2025-01-01', 'valor': 16.8}


## Validação inline

A célula final falha se o contrato, os metadados, a convenção C15, a unicidade dos anos, a
ordem, a finitude dos valores, a cobertura, os canários históricos ou a igualdade com o cálculo
forem violados. Os canários cobrem o primeiro ponto (1995), os extremos (2013, máximo; 2017,
mínimo) e os choques de 2003 e 2020; não há canário no último ponto, que muda a cada divulgação.

A validação também confere o PIB do IPEADATA contra o SIDRA 6784 em todos os anos sobrepostos
e que o valor publicado é exatamente a taxa arredondada.

In [5]:
CHAVES = {
    "versao", "id", "unidade", "eixo", "medida", "granularidade", "fonte", "dados",
}
assert set(serie) == CHAVES, sorted(set(serie) ^ CHAVES)
assert serie["versao"] == 1, serie["versao"]
assert serie["id"] == "taxa-investimento-pib", serie["id"]
assert serie["unidade"] == {"tipo": "percentual", "simbolo": "%", "casas": 1}, serie["unidade"]
assert serie["eixo"] == "percentual", serie["eixo"]
assert serie["medida"] == "participacao", serie["medida"]
assert serie["granularidade"] == "anual", serie["granularidade"]
assert serie["fonte"] == FONTE, serie["fonte"]

datas = [ponto["data"] for ponto in serie["dados"]]
por_ano = {ponto["data"][:4]: ponto["valor"] for ponto in serie["dados"]}

assert datas, "série vazia"
assert datas == sorted(datas), "datas fora de ordem"
assert all(a < b for a, b in zip(datas, datas[1:])), "datas não estritamente ascendentes"
assert datas[0] == "1995-01-01", datas[0]
assert datas[-1] == f"{ANOS[-1]}-01-01", datas[-1]
assert len(datas) == len(ANOS), (len(datas), len(ANOS))

for ponto in serie["dados"]:
    assert set(ponto) == {"data", "valor"}, sorted(ponto)
    assert isinstance(ponto["data"], str), ponto
    assert len(ponto["data"]) == 10 and ponto["data"][4:] == "-01-01", ponto
    assert not isinstance(ponto["valor"], bool), ponto
    assert isinstance(ponto["valor"], (int, float)), ponto
    assert math.isfinite(ponto["valor"]), ponto

# C15 e cobertura: um ponto por ano, de 1995 ao último ano fechado, sem lacuna.
dias = [int(data[:4]) for data in datas]
assert dias == list(range(1995, ANOS[-1] + 1)), dias

# O publicado é a taxa arredondada, ponto a ponto.
esperado = [{"data": f"{ano:04d}-01-01", "valor": arredondar(TAXA[ano])} for ano in ANOS]
assert serie["dados"] == esperado, "valor publicado difere do cálculo"

# Canários históricos fixos (não dependem da próxima divulgação).
assert por_ano["1995"] == 20.3, por_ano["1995"]
assert por_ano["2003"] == 16.6, por_ano["2003"]
assert por_ano["2013"] == 20.9, por_ano["2013"]
assert por_ano["2017"] == 14.6, por_ano["2017"]
assert por_ano["2020"] == 16.6, por_ano["2020"]

# A taxa é uma proporção: tem de ficar entre 0 e 100 e o máximo tem de ser 2013.
assert all(0 < valor < 100 for valor in por_ano.values()), por_ano
assert max(serie["dados"], key=lambda ponto: ponto["valor"])["data"] == "2013-01-01"
assert min(serie["dados"], key=lambda ponto: ponto["valor"])["data"] == "2017-01-01"

with ARQUIVO.open(encoding="utf-8") as arquivo:
    salvo = json.load(arquivo)
assert salvo == serie, "arquivo gravado difere do objeto validado"

print(
    f"ok: {len(datas)} anos únicos, {datas[0]} a {datas[-1]}, "
    f"taxa = FBCF/PIB arredondada a 1 casa; "
    f"PIB conferido contra o SIDRA em {len(conferidos)} anos"
)

ok: 31 anos únicos, 1995-01-01 a 2025-01-01, taxa = FBCF/PIB arredondada a 1 casa; PIB conferido contra o SIDRA em 29 anos
